# Training Mechanics — How a Model Actually Learns

In [ ]:
# If you are running this on Google Colab, uncomment and run the line below first.
# !pip install -q torch numpy matplotlib

## What we mean by "training"

A freshly initialised model knows nothing. Its weights — the millions or billions of numbers that determine its behaviour — start as random values. At that point, ask it to predict the next word and it will guess randomly. Give it a sentence to classify and it will be wrong most of the time.

Training is the process of adjusting those weights, step by step, until the model gets consistently better at the task.

Here is the cycle that repeats billions of times:

1. Feed the model some input
2. It produces an output (a prediction)
3. Compare the prediction to the correct answer — calculate the **loss** (how wrong it was)
4. Work out which weights contributed most to the error — this is **backpropagation**
5. Nudge those weights in the direction that reduces the error — this is the **optimiser**
6. Repeat

This notebook covers the practical machinery behind steps 3 to 5 — loss functions, gradients, optimisers, and learning rate schedules. These are the things that determine whether training converges smoothly or goes completely off the rails.

**The analogy for the whole thing:** Imagine you are learning to throw darts. You throw one, see how far it missed the bullseye (loss), figure out whether you were throwing too hard or aiming too far left (gradient), and adjust your technique slightly (optimiser). You do not overhaul your entire stance after one bad throw — you make a small correction. Repeat ten thousand times and you become good. That is exactly what model training does.

## 1. The Loss Function — measuring how wrong the model is

The loss function is a single number that summarises how bad the model's prediction was. The lower the loss, the better the prediction. The goal of training is to make this number as small as possible.

For language models doing next-token prediction, the standard loss is **cross-entropy loss**. You saw this briefly in the Learning Objectives notebook. Here is the intuition:

- The model outputs a probability for every word in the vocabulary
- Cross-entropy measures how much probability the model assigned to the *correct* word
- If the model was confident and correct — low loss
- If the model was confident and wrong — very high loss (heavily penalised)
- If the model was uncertain — medium loss

**The analogy:** Think of a weather forecaster. If they say "90% chance of rain" and it rains — great call, low penalty. If they say "90% chance of rain" and the sun blazes all day — terrible call, high penalty. If they hedge and say "50/50" — medium penalty either way. Cross-entropy rewards confident correct predictions and punishes confident wrong ones.

In [1]:
import torch
import torch.nn.functional as F

# Imagine a tiny vocabulary: [cat, dog, sat, ran, the]
vocab = ["cat", "dog", "sat", "ran", "the"]

# Three scenarios — model's confidence vs correct answer
scenarios = [
    ("Confident and correct",  torch.tensor([0.02, 0.02, 0.90, 0.03, 0.03]), 2),  # 'sat' is correct
    ("Uncertain",              torch.tensor([0.20, 0.20, 0.22, 0.19, 0.19]), 2),
    ("Confident and wrong",    torch.tensor([0.90, 0.02, 0.03, 0.02, 0.03]), 2),  # model bets on 'cat'
]

print(f"Correct next word: 'sat' (index 2)\n")
print(f"{'Scenario':<26}  {'P(correct word)':>16}  {'Loss':>8}")
print("-" * 56)

for name, probs, target_idx in scenarios:
    target = torch.tensor([target_idx])
    logits = torch.log(probs).unsqueeze(0)   # log probs as logits
    loss   = F.nll_loss(logits, target)
    print(f"  {name:<24}  {probs[target_idx].item():>15.2%}  {loss.item():>8.4f}")

print()
print("Confident and correct → loss near 0. The model is rewarded.")
print("Confident and wrong   → loss is high. The model is punished hard.")
print("This asymmetry is what drives the model to become calibrated.")

Correct next word: 'sat' (index 2)

Scenario                     P(correct word)      Loss
--------------------------------------------------------
  Confident and correct              90.00%    0.1054
  Uncertain                          22.00%    1.5141
  Confident and wrong                 3.00%    3.5066

Confident and correct → loss near 0. The model is rewarded.
Confident and wrong   → loss is high. The model is punished hard.
This asymmetry is what drives the model to become calibrated.


## 2. Gradients — figuring out who is to blame

Once we have the loss, we need to figure out which weights contributed to the error — and by how much. This is what **backpropagation** computes.

The **gradient** of the loss with respect to a weight tells you two things:
- The **sign** — should this weight go up or down to reduce the loss?
- The **magnitude** — how much does changing this weight affect the loss?

**The analogy:** Imagine you are hiking in foggy mountains and trying to find the lowest valley (lowest loss). You cannot see far ahead. But you can feel the slope under your feet right now — is it going uphill to the left and downhill to the right? The gradient is that slope reading. You take a small step in the downhill direction. Then take another reading. Repeat until you reach the bottom.

This is called **gradient descent** — following the slope of the loss surface downhill, step by step.

The entire training process is essentially: compute loss → compute gradients → take a downhill step → repeat.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Visualise a simple loss surface — one weight, one loss curve
# In a real model there are billions of weights, but the idea is the same

w_vals = torch.linspace(-3, 3, 200)
# Pretend the loss as a function of this one weight looks like this
loss_vals = (w_vals - 1.2) ** 2 + 0.5   # minimum at w=1.2

# Starting point — random initialisation
w_current = torch.tensor([-2.5], requires_grad=True)

# Simulate a few gradient descent steps
lr = 0.4
path_w, path_l = [], []

for step in range(8):
    loss = (w_current - 1.2) ** 2 + 0.5
    path_w.append(w_current.item())
    path_l.append(loss.item())
    loss.backward()
    with torch.no_grad():
        w_current -= lr * w_current.grad
    w_current.grad.zero_()

plt.figure(figsize=(9, 4))
plt.plot(w_vals.numpy(), loss_vals.numpy(), color='steelblue', linewidth=2, label='Loss surface')
plt.scatter(path_w, path_l, color='tomato', zorder=5, s=80, label='Training steps')
for i in range(len(path_w) - 1):
    plt.annotate('', xy=(path_w[i+1], path_l[i+1]), xytext=(path_w[i], path_l[i]),
                 arrowprops=dict(arrowstyle='->', color='tomato', lw=1.5))
plt.annotate('Start\n(random init)', (path_w[0], path_l[0]),
             textcoords='offset points', xytext=(-60, 10), fontsize=9, color='gray')
plt.annotate('Minimum\n(well trained)', (1.2, 0.5),
             textcoords='offset points', xytext=(15, 15), fontsize=9, color='steelblue')
plt.xlabel('Weight value')
plt.ylabel('Loss')
plt.title('Gradient Descent — following the slope downhill to the minimum')
plt.legend()
plt.tight_layout()
plt.show()

print("Each red dot is one training step.")
print("The model starts far from the minimum and gradually rolls down to it.")
print("Real models have billions of weights — same idea, just in very high dimensions.")

## 3. Optimisers — how exactly to take that downhill step

Plain gradient descent says: move each weight in the direction of the gradient by some fixed amount. Simple, but it has problems in practice.

Over decades, researchers built smarter update rules — called **optimisers** — that converge faster and more reliably.

### SGD — the original, naive approach
Just subtract the gradient times a learning rate. Works but is slow and sensitive to the learning rate choice.

### Adam — what almost everyone uses today
Adam (Adaptive Moment Estimation) is the standard optimiser for most deep learning. It does two clever things:

- **Momentum** — it keeps a running average of recent gradients. If gradients keep pointing the same direction, it accelerates. If they flip back and forth, it slows down. Like a ball rolling downhill that picks up speed on consistent slopes.
- **Adaptive learning rates** — each weight gets its own effective learning rate, automatically tuned based on how large its gradients have been historically. Weights that get noisy gradients are updated cautiously. Weights with consistent gradients move faster.

### AdamW — Adam with a fix
AdamW is Adam with a small but important correction. Adam has a subtle bug where the weight decay (a regularisation technique that prevents weights from growing too large) interacts poorly with the adaptive learning rates. AdamW fixes this. It is what most modern LLMs are trained with.

**The analogy:** SGD is like walking downhill with your eyes fixed only on the ground directly in front of you. Adam is like an experienced mountain hiker — they remember which direction has been consistently downhill, build up speed on long descents, and slow down when the path gets unpredictable.

In [ ]:
import torch
import matplotlib.pyplot as plt

def run_optimiser(optimiser_class, lr, steps=40, **kwargs):
    # A simple 2D loss surface: f(x, y) = x^2 + 5*y^2
    # Minimum is at (0, 0), but y axis is much steeper
    params = torch.tensor([-2.5, 1.8], requires_grad=True)
    opt    = optimiser_class([params], lr=lr, **kwargs)
    path   = []

    for _ in range(steps):
        path.append(params.detach().clone().numpy().copy())
        opt.zero_grad()
        loss = params[0]**2 + 5 * params[1]**2
        loss.backward()
        opt.step()

    return path

import numpy as np
paths = {
    "SGD  (lr=0.1)" : run_optimiser(torch.optim.SGD,  lr=0.1),
    "Adam (lr=0.3)" : run_optimiser(torch.optim.Adam, lr=0.3),
    "AdamW(lr=0.3)" : run_optimiser(torch.optim.AdamW, lr=0.3, weight_decay=0.01),
}
colors = ["tomato", "steelblue", "seagreen"]

# Loss surface contours
x = np.linspace(-3, 3, 200)
y = np.linspace(-2.5, 2.5, 200)
X, Y = np.meshgrid(x, y)
Z = X**2 + 5 * Y**2

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, path), color in zip(axes, paths.items(), colors):
    ax.contour(X, Y, Z, levels=15, cmap='Blues', alpha=0.5)
    px = [p[0] for p in path]
    py = [p[1] for p in path]
    ax.plot(px, py, color=color, linewidth=1.5, marker='o', markersize=3)
    ax.scatter(px[0], py[0], color='black', s=80, zorder=5, label='Start')
    ax.scatter(0, 0, color='gold', s=100, zorder=5, marker='*', label='Minimum')
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('w₁')
    ax.set_ylabel('w₂')
    ax.legend(fontsize=8)

plt.suptitle('Optimiser paths on the same loss surface', fontsize=12)
plt.tight_layout()
plt.show()

print("SGD can oscillate or take a winding path on surfaces with different curvature in each direction.")
print("Adam and AdamW adapt their step size per dimension — smoother, faster convergence.")

## 4. The Learning Rate — the single most important hyperparameter

The learning rate controls how big each update step is. It is, without question, the hyperparameter that most affects whether training succeeds or fails.

**Too high:** The model overshoots the minimum. Instead of rolling smoothly into the valley, it bounces off the sides and might fly out entirely. Loss diverges — you will see it spike upward and never recover.

**Too low:** The model takes tiny baby steps. It will eventually get there, but training takes forever. Or it gets stuck in a local minimum early on.

**Just right:** Loss decreases steadily and the model converges.

**The analogy:** The learning rate is like the volume knob on a guitar amplifier. Turn it too high and everything distorts and buzzes — nothing useful comes out. Turn it too low and nobody can hear you. Find the sweet spot and the music flows.

A rule of thumb that works surprisingly often: start with `1e-3` for Adam/AdamW, and if things explode try `1e-4`. If things are learning but very slowly, try `3e-3`.

In [ ]:
import torch
import matplotlib.pyplot as plt

def train_with_lr(lr, steps=60):
    w    = torch.tensor([-2.5], requires_grad=True)
    opt  = torch.optim.SGD([w], lr=lr)
    losses = []
    for _ in range(steps):
        opt.zero_grad()
        loss = (w - 1.0) ** 2
        losses.append(loss.item())
        loss.backward()
        opt.step()
    return losses

configs = [
    (0.01,  "Too low  (lr=0.01)",  "#aab4c8"),
    (0.5,   "Just right (lr=0.5)", "seagreen"),
    (1.95,  "Too high  (lr=1.95)", "tomato"),
]

plt.figure(figsize=(10, 4))
for lr, label, color in configs:
    losses = train_with_lr(lr)
    plt.plot(losses, label=label, color=color, linewidth=2)

plt.xlabel('Training step')
plt.ylabel('Loss')
plt.title('Effect of learning rate on training')
plt.legend()
plt.ylim(-0.5, 15)
plt.tight_layout()
plt.show()

print("Too low  → loss decreases very slowly, training takes ages.")
print("Just right → loss drops quickly and smoothly to near zero.")
print("Too high → loss bounces wildly or diverges completely.")

## 5. Learning Rate Schedules — not staying at the same speed the whole time

Using a fixed learning rate throughout training is not optimal. What works well at the start of training often works poorly near the end.

At the start, the model knows nothing and needs to take bigger steps to move quickly toward a good region. Near the end, the model is close to a good solution and needs smaller, careful steps to settle precisely without overshooting.

This is why real training runs use a **learning rate schedule** — a plan for how the learning rate changes over time.

**The analogy:** Think about how you drive to an unfamiliar destination. At the start, you are on the motorway — you go fast. As you get into the city streets, you slow down. As you approach the exact address, you slow to a crawl looking for the right house. You do not drive at motorway speed the whole way — you would overshoot and crash.

### Warmup
At the very beginning of training, gradients are noisy and unreliable because the model weights are random. Starting with a large learning rate on random weights causes chaotic updates. **Warmup** starts the learning rate near zero and ramps it up gradually over the first few thousand steps — giving the model time to stabilise before the full learning rate kicks in.

### Cosine Decay
After warmup, the most common schedule today is **cosine decay** — the learning rate follows a cosine curve from its peak down to near zero by the end of training. It decays quickly at first, then slows down as it approaches zero, which mirrors how training progress works — fast early gains, fine-tuning at the end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def lr_schedule(step, total_steps, peak_lr, warmup_steps):
    if step < warmup_steps:
        # Linear warmup from 0 to peak_lr
        return peak_lr * (step / warmup_steps)
    else:
        # Cosine decay from peak_lr to near 0
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return peak_lr * 0.5 * (1 + np.cos(np.pi * progress))

total_steps  = 1000
warmup_steps = 100
peak_lr      = 3e-4

steps = np.arange(total_steps)
lrs   = [lr_schedule(s, total_steps, peak_lr, warmup_steps) for s in steps]

plt.figure(figsize=(10, 4))
plt.plot(steps, lrs, color='steelblue', linewidth=2)
plt.axvline(warmup_steps, color='tomato', linestyle='--', alpha=0.7, label=f'Warmup ends (step {warmup_steps})')
plt.fill_betweenx([0, peak_lr], 0, warmup_steps, alpha=0.08, color='tomato', label='Warmup region')
plt.fill_betweenx([0, peak_lr], warmup_steps, total_steps, alpha=0.08, color='steelblue', label='Cosine decay region')
plt.xlabel('Training step')
plt.ylabel('Learning rate')
plt.title('Warmup + Cosine Decay schedule — the standard for modern LLM training')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Peak learning rate : {peak_lr}")
print(f"Warmup             : first {warmup_steps} steps ramp from 0 → {peak_lr}")
print(f"Cosine decay       : steps {warmup_steps}–{total_steps} decay from {peak_lr} → ~0")
print()
print("GPT-3 used this exact schedule. Most open source LLMs (LLaMA, Mistral) do too.")

## 6. Gradient Clipping — the emergency brake

Occasionally during training, the gradients can suddenly become enormous — a single batch of unusual data causes the loss to spike, the gradients explode, and the weights get a massive update that ruins everything the model had learned up to that point.

This is called the **exploding gradient problem**, and it is particularly common when training very deep networks or on sequences that are unexpectedly long.

**Gradient clipping** is the fix. Before each weight update, you check the total size (norm) of the gradient vector. If it exceeds a threshold, you scale it down to exactly the threshold. The direction is preserved — you are still going downhill — but the step is capped so it cannot be catastrophically large.

**The analogy:** Imagine you are cycling downhill. Normally you pedal gently and it is fine. But on a steep hill, you pick up speed fast and you need the brakes. Gradient clipping is the brake — it does not stop you going downhill, it just prevents you from reaching a speed where you lose control.

In PyTorch: `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`

The value `1.0` is the standard for LLM training. Almost every training recipe uses it.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)

# Simulate a gradient vector with a very large norm (exploding gradient)
grad = torch.tensor([8.5, -6.2, 12.1, -9.4, 3.7])
max_norm = 1.0

grad_norm_before = grad.norm().item()

# Clipping — scale down if norm exceeds max_norm
clipped_grad = grad.clone()
torch.nn.utils.clip_grad_norm_([clipped_grad], max_norm=max_norm)

# Manual version to show the math clearly
scale = min(1.0, max_norm / grad_norm_before)
manual_clipped = grad * scale

grad_norm_after = clipped_grad.norm().item()

print(f"Original gradient  : {grad.tolist()}")
print(f"Gradient norm      : {grad_norm_before:.4f}  ← way too large")
print()
print(f"After clipping     : {manual_clipped.tolist()}")
print(f"Clipped norm       : {grad_norm_after:.4f}  ← capped at max_norm={max_norm}")
print()
print("Direction is preserved — the model still moves in the right direction.")
print("Only the magnitude is capped — no catastrophic update.")

# Show direction preserved
cos_sim = torch.dot(grad, manual_clipped) / (grad.norm() * manual_clipped.norm())
print(f"\nCosine similarity between original and clipped gradient: {cos_sim.item():.4f}")
print("1.0 = exactly the same direction — clipping only scales, never rotates.")

## Putting it all together — a tiny training loop

Let us wire every concept from this notebook into a single, readable training loop. This is the skeleton that sits inside every model training script — from a toy example to GPT-3.

The model here is trivially small (one linear layer). The concepts are identical.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

# --- tiny dataset: learn to map x → 2x + 1 ---
X = torch.randn(200, 1)
y = 2 * X + 1 + torch.randn(200, 1) * 0.1   # slight noise

# --- model: one linear layer ---
model = nn.Linear(1, 1)

# --- optimiser: AdamW (the LLM standard) ---
optimiser = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=0.01)

# --- learning rate schedule: warmup then cosine decay ---
total_steps  = 300
warmup_steps = 30

def get_lr(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimiser, get_lr)

# --- training loop ---
losses = []
lrs    = []

for step in range(total_steps):
    optimiser.zero_grad()

    pred = model(X)
    loss = F.mse_loss(pred, y)

    loss.backward()

    # Gradient clipping — the emergency brake
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimiser.step()
    scheduler.step()

    losses.append(loss.item())
    lrs.append(scheduler.get_last_lr()[0] * 1e-2)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(losses, color='tomato', linewidth=1.5)
ax1.set_ylabel('Loss')
ax1.set_title('Training loss over time')
ax1.grid(True, alpha=0.3)

ax2.plot(lrs, color='steelblue', linewidth=1.5)
ax2.set_ylabel('Learning rate')
ax2.set_xlabel('Step')
ax2.set_title('Learning rate schedule (warmup + cosine decay)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

w = model.weight.item()
b = model.bias.item()
print(f"Learned: y = {w:.4f}x + {b:.4f}")
print(f"Target : y = 2.0000x + 1.0000")
print()
print("Every LLM training script has this same skeleton:")
print("  zero_grad → forward pass → compute loss → backward → clip → step → schedule")

## Key takeaways

- **Loss function** measures how wrong the model is. Lower is better. Cross-entropy is standard for language models — it rewards confident correct predictions and heavily punishes confident wrong ones.
- **Gradients** tell you which direction to move each weight to reduce the loss. Backpropagation computes them automatically.
- **Gradient descent** follows the gradient downhill — small step by small step — until the model reaches a low-loss region.
- **AdamW** is the standard optimiser for modern LLMs. It adapts the learning rate per weight and handles weight decay correctly.
- **Learning rate** is the most important hyperparameter. Too high and training diverges. Too low and it takes forever. `1e-3` or `3e-4` with AdamW is a reliable starting point.
- **Warmup + cosine decay** is the standard schedule. Ramp up slowly at first to stabilise, then decay gracefully to the end.
- **Gradient clipping** (`max_norm=1.0`) is the emergency brake. It caps catastrophically large gradient updates while preserving the update direction.
- The training loop is always the same skeleton: `zero_grad → forward → loss → backward → clip → step → schedule`.

---

You have now covered all five foundational topics. You know how text becomes numbers (tokenization), how those numbers carry meaning (embeddings), how models focus on context (attention), how the pieces assemble into an architecture (transformer internals), what the model is trained to do (learning objectives), and how that training actually works (training mechanics).

From here, everything else in this repo — RAG, agents, fine-tuning, evaluation — builds on these foundations.